In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr
from adjustText import adjust_text
import gseapy as gp
from gseapy import barplot, dotplot
import numpy as np
from functools import reduce
from scipy.stats import combine_pvalues
from statsmodels.stats.multitest import multipletests
import scanpy as sc
import anndata
from src.eval import calculate_results_per_batch
import torch

sns.set_style('white')
sns.set_context("paper", font_scale = 1.2)

out_dir = '/mlbio_scratch/vinas/image2transcriptome/results'
fig_dir = f'{out_dir}/iris_figures'

/home/vinas/.virtualenvs/iris/lib/python3.7/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
whitelist_df = pd.read_csv(f'{fig_dir}/cid_whitelist.csv')
toremove_df = pd.read_csv(f'{fig_dir}/cid_nuc_toremove.csv')
whitelist_ids = np.array(whitelist_df.columns)
remove_cell_ids = np.array(toremove_df.columns)

In [3]:
seed = 2
model_path = 'fucci_3t3_221124_filtered_noNG030JP208_405-bg-random_resnet'
path = out_dir + '/' + model_path + '/seed_{}/test_preds.h5ad'
test_adata = anndata.read_h5ad(path.format(seed))

In [4]:
corrs = []
keep_idxs = np.setdiff1d(test_adata.obs.index, remove_cell_ids)
test_adata_ = test_adata[keep_idxs]
unique_exps = test_adata_.obs['exp_id'].unique()
stds = {}
for exp_id in unique_exps:
    test_adata_exp = test_adata_[test_adata_.obs['exp_id'] == exp_id]
    stds[exp_id] = np.array(np.std(test_adata_exp.X, axis=0))

#### Aggregate results

In [5]:
seeds = [0, 1, 2]

model_paths = [
    # 'fucci_3t3_221124_filtered_noNG030JP208_seg-bg-random/size_model',
    # 'fucci_3t3_221124_filtered_noNG030JP208_488-561-seg-bg-random/mlp_fucci_angle_model',
    # 'fucci_3t3_221124_filtered_noNG030JP208_405-bg-random_resnet',
    'fucci_3t3_221124_filtered_noNG030JP208_seg-bg-random_resnet'
]

for model_path in model_paths:
    print(model_path)
    for seed in seeds:
        path = out_dir + '/' + model_path + '/seed_{}/test_preds.h5ad'
        test_adata = anndata.read_h5ad(path.format(seed))
        keep_idxs = np.intersect1d(test_adata.obs.index, whitelist_ids) # np.setdiff1d(test_adata.obs.index, remove_cell_ids)
        test_adata_ = test_adata[keep_idxs]
        print(test_adata_.shape, test_adata.shape)
        results_df = calculate_results_per_batch(test_adata_, test_adata_.layers['pred'], test_adata_.X)
        results_df.to_csv(out_dir + '/' + model_path + '/seed_{}/'.format(seed) + 'test_results_qc.csv')

fucci_3t3_221124_filtered_noNG030JP208_seg-bg-random_resnet
(1037, 2000) (1248, 2000)
(1037, 2000) (1248, 2000)
(1046, 2000) (1248, 2000)


In [6]:
def calculate_consensus(path_template, seeds, alpha=0.05, metric='avg_Pearson'):
    results_dfs = []
    pvals_dfs = []
    for seed in seeds:
        # print(path_template.format(seed=seed))
        results_df = pd.read_csv(path_template.format(seed), index_col='genes')
        results_df[f'{metric}_{seed}'] = results_df[metric]

        # Combine p-values across batches
        test_batches = [s.split('_')[1] for s in results_df.columns if s.startswith('Pearson_')]
        batch_pval_columns = [f'pvals_{s}' for s in test_batches]
        pvalues = []
        for g, row in results_df[batch_pval_columns].iterrows():
            combined_pval = combine_pvalues(row.values, method='fisher')[1]
            pvalues.append(combined_pval)
        results_df[f'{metric}_pval'] = pvalues
        pvals_dfs.append(results_df[f'{metric}_pval'])
        results_dfs.append(results_df[f'{metric}_{seed}'])
    
    # Average results
    merged_df = reduce(lambda left, right: pd.merge(left, right, left_index=True, right_index=True), results_dfs)
    merged_df[metric] = merged_df.values.mean(axis=-1)

    # Combine p-values across seeds
    genes = merged_df.index
    combined_pvals_df = pd.DataFrame(index=genes, columns=['combined_pval'])
    for g in genes:
        pvals = [df.loc[g] for df in pvals_dfs]
        combined_pvals_df.loc[g] = combine_pvalues(pvals, method='fisher')[1]
    merged_df[f'{metric}_pval'] = combined_pvals_df[f'combined_pval']
    merged_df[f'{metric}_fdr'] =  multipletests(combined_pvals_df[f'combined_pval'], alpha=alpha, method='fdr_bh')[1]
    # false_discovery_control(combined_pvals_df[f'combined_pval'])
    return merged_df

def plot_corrs(hoechst_corr, bf_corr,
               pvals1=None,
               pvals2=None,
               pval_threshold=0.05,
               min_corr=0.1,
               k_h = 20,
               k_bf = 20,
               cutoff_bf_high = 0.2,
               cutoff_h_high = 0.2,
               text_size=10,
               mod1='Hoechst',
               mod2='Brightfield',
               metric='Pearson correlation',
               highlight_genes=None
               ):
    if pvals1 is None or pvals2 is None:
        plt.scatter(hoechst_corr, bf_corr, s=3, c='lightgray')
    else:
        pval_mask = np.logical_or(pvals1 < pval_threshold, pvals2 < pval_threshold)
        plt.scatter(hoechst_corr[pval_mask], bf_corr[pval_mask], s=5, c='darkgray')
        plt.scatter(hoechst_corr[~pval_mask], bf_corr[~pval_mask], s=3, c='lightgray')
    plt.xlabel(f'Test {metric} ({mod1})')
    plt.ylabel(f'Test {metric} ({mod2})');

    top_h_genes = np.argsort(hoechst_corr.values - np.maximum(bf_corr.values, 0))[:k_h]
    top_bf_genes = np.argsort(bf_corr.values - np.maximum(hoechst_corr.values, 0))[:k_bf]

    texts = []
    for i, (x, y, l) in enumerate(zip(hoechst_corr, bf_corr, hoechst_corr.index)):
        if (highlight_genes is not None and l in highlight_genes):
            texts.append(plt.text(x, y, l, size=text_size, ha='center', va='center'))
            plt.scatter(x, y, s=10, c='purple')
        elif (x > cutoff_h_high and y > cutoff_bf_high):
            texts.append(plt.text(x, y, l, size=text_size, ha='center', va='center'))
            plt.scatter(x, y, s=10, c='green')
        elif i in top_bf_genes and x > min_corr:
            texts.append(plt.text(x, y, l, size=text_size, ha='center', va='center'))
            plt.scatter(x, y, s=10, c='orange')
        elif i in top_h_genes and y > min_corr:
            texts.append(plt.text(x, y, l, size=text_size, ha='center', va='center'))
            plt.scatter(x, y, s=10, c='blue')

    # plt.xlim(-0.25, None)
    # plt.ylim(-0.25, None)
    ax = plt.gca()
    min_ = max(bf_corr.min(), hoechst_corr.min())
    max_ = max(bf_corr.max(), hoechst_corr.max())
    ax.plot([min_, max_], [min_, max_], ls='--', color='black', alpha=0.25)
    # Hide the right and top spines
    ax.spines[['right', 'top']].set_visible(False)
    plt.title(f'Performance by channel ({metric})')
    # plt.grid(linestyle='dotted')
    adjust_text(texts, arrowprops=dict(arrowstyle="-", color='k', lw=0.5))
    print('Corr: ', pearsonr(hoechst_corr, bf_corr))

In [7]:
seeds = [0, 1, 2]
filename = 'test_results_qc.csv'

size_df = calculate_consensus(out_dir + '/fucci_3t3_221124_filtered_noNG030JP208_seg-bg-random/size_model/seed_{}/'+filename, seeds, metric='avg_Pearson')
size_df['channel'] = 'Size'

# fucci_df = calculate_consensus(out_dir + '/fucci_3t3_221124_filtered_noNG030JP208_488-561-seg-bg-random/fucci_angle_model/seed_{}/test_results.csv', seeds, metric='avg_Pearson')
fucci_df = calculate_consensus(out_dir + '/fucci_3t3_221124_filtered_noNG030JP208_488-561-seg-bg-random/mlp_fucci_angle_model/seed_{}/'+filename, seeds, metric='avg_Pearson')
fucci_df['channel'] = 'FUCCI angle'

fucci405_merged_df = calculate_consensus(out_dir +'/fucci_3t3_221124_filtered_noNG030JP208_405-bg-random_resnet/seed_{}/'+filename, seeds)
fucci405_merged_df['channel'] = 'Hoechst'

fucci_seg_merged_df = calculate_consensus(out_dir +'/fucci_3t3_221124_filtered_noNG030JP208_seg-bg-random_resnet/seed_{}/'+filename, seeds)
fucci_seg_merged_df['channel'] = 'Segmentation'

all_merged_df = pd.concat([size_df, fucci_seg_merged_df, fucci_df, fucci405_merged_df], axis=0)
all_merged_df = all_merged_df[['avg_Pearson', 'avg_Pearson_pval', 'avg_Pearson_fdr', 'channel']]

In [8]:
fucci488_561_merged_ccgo_df = calculate_consensus(out_dir +'/fucci_3t3_221124_filtered_noNG030JP208_488-561-random_resnet_CCGO/seed_{}/test_results.csv', seeds)
fucci488_561_merged_tirosh_df = calculate_consensus(out_dir +'/fucci_3t3_221124_filtered_noNG030JP208_488-561-random_resnet_tirosh/seed_{}/test_results.csv', seeds)

cc_genes_intersect = np.intersect1d(fucci488_561_merged_ccgo_df.index, fucci488_561_merged_tirosh_df.index)
cc_genes = np.union1d(fucci488_561_merged_ccgo_df.index, fucci488_561_merged_tirosh_df.index)
all_merged_df['Cell cycle'] = [g in cc_genes_intersect for g in all_merged_df.index]
all_merged_df

,avg_Pearson,avg_Pearson_pval,avg_Pearson_fdr,channel,Cell cycle
genes,,,,,
Prc1,0.329203,0.0,0.0,Size,False
Top2a,0.310587,0.0,0.0,Size,True
Nusap1,0.285513,0.0,0.0,Size,True
Cdk1,0.262361,0.0,0.0,Size,True
Spc25,0.241991,0.0,0.0,Size,False
...,...,...,...,...,...
Jam2,-0.020440,0.40692,0.679589,Hoechst,False
Magi2,-0.014645,0.076094,0.24497,Hoechst,False
Slc25a30,-0.033829,0.145803,0.376165,Hoechst,False


In [9]:
# all_merged_df_ = pd.merge(all_merged_df, adata.var, left_index=True, right_index=True)
df_wide = all_merged_df.reset_index().pivot(index='genes', columns='channel', values=['avg_Pearson', 'avg_Pearson_pval', 'avg_Pearson_fdr'])

gene_mask = df_wide['avg_Pearson'].isna().any(axis=1)
df_wide = df_wide.loc[~gene_mask]
df_wide['Cell cycle'] = [g in cc_genes for g in df_wide.index]
# df_wide['avg_batchmean'] = [adata.var.loc[g, 'mean'] if g in adata.var.index else np.nan for g in df_wide.index]
# df_wide['avg_batchsd'] = [adata.var.loc[g, 'std'] if g in adata.var.index else np.nan for g in df_wide.index]
df_wide.to_csv(f'{fig_dir}/all_channels_results_qc_clean.csv')
df_wide

avg_Pearson                                  avg_Pearson_pval  \
channel       FUCCI angle   Hoechst Segmentation      Size      FUCCI angle   
genes                                                                         
1110032A03Rik   -0.020642 -0.013767     0.007666 -0.004886         0.987459   
1190007I07Rik    0.016679  0.028992     0.022723  0.053113         0.984212   
1500026H17Rik    0.035788 -0.012633    -0.007771 -0.015324         0.236336   
1600010M07Rik    0.058833  0.050859     0.071041  0.123473         0.245644   
1600019K03Rik    0.109234  0.141019     0.192132   0.15235         0.000001   
...                   ...       ...          ...       ...              ...   
mt-Rnr2          0.193563   0.10524     0.198858  0.232408              0.0   
mt-Tf            0.013685 -0.000034    -0.002509 -0.010932         0.131027   
mt-Tm           -0.003512 -0.017922     0.017419   0.00786         0.971044   
mt-Tp           -0.014608 -0.025788    -0.003025  0.017764         0.303483   
n-R5-8s1         0.072208  0.071858     0.110094  0.074268         0.015405   

                                               avg_Pearson_fdr            \
channel         Hoechst Segmentation      Size     FUCCI angle   Hoechst   
genes                                                                      
1110032A03Rik  0.325343     0.812909  0.694411        0.990834  0.600759   
1190007I07Rik  0.395384     0.285034  0.064012        0.990081  0.666388   
1500026H17Rik  0.481543     0.727656  0.917647        0.425173  0.745372   
1600010M07Rik  0.018407     0.006751  0.000001        0.436041  0.083917   
1600019K03Rik       0.0          0.0       0.0        0.000004       0.0   
...                 ...          ...       ...             ...       ...   
mt-Rnr2        0.000001          0.0       0.0             0.0  0.000008   
mt-Tf          0.383244     0.202117  0.414907        0.284117  0.658949   
mt-Tm          0.549562     0.347445  0.835959        0.982797  0.791155   
mt-Tp          0.022603     0.661836  0.425404        0.498336  0.095901   
n-R5-8s1       0.000212          0.0  0.016144        0.051748  0.001823   

                                     Cell cycle  
channel       Segmentation      Size             
genes                                            
1110032A03Rik     0.892735   0.82732      False  
1190007I07Rik     0.486678  0.168103      False  
1500026H17Rik     0.844512  0.955394      False  
1600010M07Rik     0.026857  0.000006      False  
1600019K03Rik          0.0       0.0      False  
...                    ...       ...        ...  
mt-Rnr2                0.0       0.0      False  
mt-Tf             0.390324  0.607261      False  
mt-Tm              0.55185  0.912407      False  
mt-Tp             0.805035  0.617088      False  
n-R5-8s1               0.0  0.060476      False  

[1468 rows x 13 columns]

In [13]:
# all_merged_df_ = pd.merge(all_merged_df, adata.var, left_index=True, right_index=True)
df_wide = all_merged_df.reset_index().pivot(index='genes', columns='channel', values=['avg_Pearson', 'avg_Pearson_pval', 'avg_Pearson_fdr'])

gene_mask = df_wide['avg_Pearson'].isna().any(axis=1)
df_wide = df_wide.loc[~gene_mask]
df_wide['Cell cycle'] = [g in cc_genes for g in df_wide.index]
# df_wide['avg_batchmean'] = [adata.var.loc[g, 'mean'] if g in adata.var.index else np.nan for g in df_wide.index]
# df_wide['avg_batchsd'] = [adata.var.loc[g, 'std'] if g in adata.var.index else np.nan for g in df_wide.index]
df_wide.to_csv(f'{fig_dir}/all_channels_results_qc_clean.csv')
df_wide

avg_Pearson                                  avg_Pearson_pval  \
channel       FUCCI angle   Hoechst Segmentation      Size      FUCCI angle   
genes                                                                         
1110032A03Rik   -0.020642 -0.013767     0.013287 -0.004886         0.987459   
1190007I07Rik    0.016679  0.028992    -0.023706  0.053113         0.984212   
1500026H17Rik    0.035788 -0.012633    -0.008166 -0.015324         0.236336   
1600010M07Rik    0.058833  0.050859     0.018485  0.123473         0.245644   
1600019K03Rik    0.109234  0.141019     0.144785   0.15235         0.000001   
...                   ...       ...          ...       ...              ...   
mt-Rnr2          0.193563   0.10524     0.165664  0.232408              0.0   
mt-Tf            0.013685 -0.000034    -0.050489 -0.010932         0.131027   
mt-Tm           -0.003512 -0.017922     0.017998   0.00786         0.971044   
mt-Tp           -0.014608 -0.025788      -0.0003  0.017764         0.303483   
n-R5-8s1         0.072208  0.071858     0.054272  0.074268         0.015405   

                                               avg_Pearson_fdr            \
channel         Hoechst Segmentation      Size     FUCCI angle   Hoechst   
genes                                                                      
1110032A03Rik  0.325343     0.369694  0.694411        0.990834  0.600759   
1190007I07Rik  0.395384     0.328478  0.064012        0.990081  0.666388   
1500026H17Rik  0.481543     0.030483  0.917647        0.425173  0.745372   
1600010M07Rik  0.018407     0.072664  0.000001        0.436041  0.083917   
1600019K03Rik       0.0          0.0       0.0        0.000004       0.0   
...                 ...          ...       ...             ...       ...   
mt-Rnr2        0.000001          0.0       0.0             0.0  0.000008   
mt-Tf          0.383244     0.023148  0.414907        0.284117  0.658949   
mt-Tm          0.549562     0.383327  0.835959        0.982797  0.791155   
mt-Tp          0.022603     0.002879  0.425404        0.498336  0.095901   
n-R5-8s1       0.000212     0.031514  0.016144        0.051748  0.001823   

                                     Cell cycle  
channel       Segmentation      Size             
genes                                            
1110032A03Rik      0.64112   0.82732      False  
1190007I07Rik     0.602004  0.168103      False  
1500026H17Rik     0.136429  0.955394      False  
1600010M07Rik     0.253021  0.000006      False  
1600019K03Rik          0.0       0.0      False  
...                    ...       ...        ...  
mt-Rnr2                0.0       0.0      False  
mt-Tf             0.113648  0.607261      False  
mt-Tm             0.644587  0.912407      False  
mt-Tp             0.023877  0.617088      False  
n-R5-8s1          0.138929  0.060476      False  

[1468 rows x 13 columns]